<a href="https://colab.research.google.com/github/ribesstefano/PROTAC-Splitter/blob/main/notebooks/trl_protac_splitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning PROTAC-Splitter in TRL

## Setup

In [1]:
!pip install datasets transformers peft accelerate bitsandbytes evaluate rouge_score huggingface_hub rdkit tensorboard -qqq -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 96.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.5/30.5 MB 40.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118

In [2]:
!git config --global user.name "ribesstefano"
!git config --global user.mail "ribes.stefano@gmail.com"
!git clone https://github.com/ribesstefano/trl.git
%cd trl
!git checkout encoder_decoder_patch
!pip install . -qqq

Cloning into 'trl'...
remote: Enumerating objects: 3508, done.
remote: Counting objects: 100% (3507/3507), done.
remote: Compressing objects: 100% (1349/1349), done.
remote: Total 3508 (delta 2222), reused 3106 (delta 1969), pack-reused 1
Receiving objects: 100% (3508/3508), 5.40 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (2222/2222), done.
/content/trl
Branch 'encoder_decoder_patch' set up to track remote branch 'encoder_decoder_patch' from 'origin'.
Switched to a new branch 'encoder_decoder_patch'
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 749.5 kB/s eta 0:00:00


In [19]:
%cd ..
!ls

/content
sample_data  trl


## TRL

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
import torch
from transformers import (
    EncoderDecoderModel,
    EncoderDecoderConfig,
    AutoTokenizer,
)
from trl import (
    AutoModelForCausalLMWithValueHead,
    AutoModelForSeq2SeqLMWithValueHead,
    PPOConfig,
    PPOTrainer,
    create_reference_model,
)

/content/trl/trl/trainer/ppo_config.py:141: UserWarning: The `optimize_cuda_cache` arguement will be deprecated soon, please use `optimize_device_cache` instead.
  warnings.warn(


### Setup Dataset

In [14]:
from datasets import load_dataset

dataset = load_dataset("ailab-bio/PROTAC-Substructures", "unlabeled", split="train")
dataset = dataset.rename_column("text", "query")
dataset = dataset.remove_columns(["labels"])
dataset

Dataset({
    features: ['query'],
    num_rows: 1763
})

### Setup Model

In [27]:
pretrained_model = "ailab-bio/PROTAC-Splitter_untied_80-20-split"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
bert2bert = EncoderDecoderModel.from_pretrained(pretrained_model)
config = EncoderDecoderConfig.from_pretrained(pretrained_model)

In [11]:
from transformers import AutoTokenizer
from trl import AutoModelForSeq2SeqLMWithValueHead, PPOConfig, PPOTrainer

model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

tokenizer.pad_token = tokenizer.eos_token

In [30]:
def tokenize(sample):
    sample["input_ids"] = tokenizer.encode(sample["query"], padding='max_length')
    return sample

train_dataset = dataset.map(tokenize, batched=False)
train_dataset

Map:   0%|          | 0/1763 [00:00<?, ? examples/s]

Dataset({
    features: ['query', 'input_ids'],
    num_rows: 1763
})

## Reward Function

## Training Loop

In [35]:
from trl import PPOConfig, PPOTrainer

ppo_config = PPOConfig(
    model_name=pretrained_model,
    learning_rate=1.41e-5,
    seed=42,
    steps=2000, # Default: 20_000
    ppo_epochs=4, # Default: 4
    is_encoder_decoder=True,
    global_batch_size=32,
)

ppo_trainer = PPOTrainer(
    model=model,
    config=ppo_config,
    tokenizer=tokenizer,
    dataset=train_dataset,
)

In [32]:
from tqdm import tqdm

generation_kwargs = {
    "do_sample": False,
    "pad_token_id": tokenizer.eos_token_id,
}

for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    #### Get response from SFTModel
    response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    #### Compute reward score
    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    print(texts)
    break
#     pipe_outputs = reward_model(texts)
#     rewards = [torch.tensor(output[1]["score"]) for output in pipe_outputs]

#     #### Run PPO step
#     stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
#     ppo_trainer.log_stats(stats, batch, rewards)

# #### Save model
# ppo_trainer.save_model("my_ppo_model")

0it [00:00, ?it/s]You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
0it [02:49, ?it/s]

['CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(CC(=O)N1CCC(C#Cc2ccc(C(=O)NC3C(C)(C)C(Oc4ccc(C#N)c(Cl)c4)C3(C)C)cc2)CC1)c1ccccc1)C(C)C<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCOCCOCCOCCOCCOCCNC(=O)CCC[*:1].[*:1]N1CCC2CCC(NC(=O)NC(CCC(Oc3ccc(C)(C)c(F)cc3F)c(Nc4ccccc4S(=O)(=O)C(N)=O)c3)ccc(-c2)cc1</s>', 'CCC(C(=O)N1CCCCC1C(=O)OC(CCc1ccc(OC)c(OC)c1)c1ccccc1OCC(=O)NCCCOCCOCCOCCCNC(=O)COc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)c1cc(OC)c(OC)c(OC)c1<s><s>[*:1]N1CCN(Cc2ccc(C(=O)Nc3ccc(C)c(C#Cc4ccc4[nH]ncc5c4)c3)cc2C(F)(F)F)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCC[*:1]</s>', 'CC1Cc2c([nH]c3ccccc23)C(c2cnc(N3CCC4(CC3)CC(CN3CC5(CCN(c6ccc7c(c6)CN(C6CCC(=O)NC6=O)C7=O)CC5)C3)C4)nc2)N1CC(C)(C)F<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCCCCCCCCCNC(=O)[*:1].[*:1]Nc1cc(C(=O)Nc2ccccc2N(C1)C1CC(C)(C)C)C.[*:2]CCCCCCCCCCNC(=O)[*:1]</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(-n2cc(-c3cncnc3)c3nc(N(C)C4CCN(C(=O)CCOCCOCCC(=O)NC(C(=O)N5CC(O)CC5C(=O)NCc5ccc(-c6scnc6C)cc5

In [3]:
import torch
from transformers import (
    EncoderDecoderModel,
    EncoderDecoderConfig,
    AutoTokenizer,
)
from trl import (
    AutoModelForCausalLMWithValueHead,
    AutoModelForSeq2SeqLMWithValueHead,
    PPOConfig,
    PPOTrainer,
    create_reference_model,
)

pretrained_model = "ribesstefano/ChemBERTa2ChemBERTa-58M"
# tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
bert2bert = EncoderDecoderModel.from_pretrained(pretrained_model)
config = EncoderDecoderConfig.from_pretrained(pretrained_model)
# print(bert2bert.encoder.config.hidden_size)
# print(bert2bert.decoder.config.hidden_size)
# print(bert2bert.config.hidden_size)

model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model)
# # model_ref = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model, device_map="auto", load_in_8bit=True)
# model_ref = create_reference_model(model, num_shared_layers=6)

# tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
# tokenizer.pad_token = tokenizer.eos_token

/content/trl/trl/trainer/ppo_config.py:141: UserWarning: The `optimize_cuda_cache` arguement will be deprecated soon, please use `optimize_device_cache` instead.
  warnings.warn(


The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']


The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
